### 3. Longest Substring Without Repeating Characters

[Link](https://leetcode.com/problems/longest-substring-without-repeating-characters/description/)

Given a string `s`, find the **length** of the **longest substring** without duplicate characters.

**Example 1**:
```shell
Input: s = "abcabcbb"
Output: 3
```
**Explanation**: The answer is `"abc"`, with the length of 3. Note that `"bca"` and `"cab"` are also correct answers.

**Example 2**:
```shell
Input: s = "bbbbb"
Output: 1
```
**Explanation**: The answer is `"b"`, with the length of 1.

**Example 3**:
```shell
Input: s = "pwwkew"
Output: 3
```
**Explanation**: The answer is `"wke"`, with the length of 3.
Notice that the answer must be a substring, `"pwke"` is a subsequence and not a substring.

**Constraints**:

- `0 <= s.length <= 5 * 104`
- `s` consists of English letters, digits, symbols and spaces.

1. [Visual](../images/3longest_substring_withou_reps.png)
2. [Neet Expanation](https://www.youtube.com/watch?v=wiGpQwVHdE0&t=400s)

### Reasoning - amortized analysis
Across the _entire_ algorithm, from the very first iteration of the outer for-loop to the very last:

1. Does `l` ever move backward, or reset to a smaller value, at any point?
2. It starts at 0. What's the largest value it could possibly reach by the time the algorithm finishes?
- `l` never moved backward nor reset to a smaller index at any point. The largest value it could possibly reach is `<= n` by the time the algorithm is done.

Exactly — `l` is monotonic, **bounded between 0 and `n`**, so it can increment **at most `n` times total**, summed across the whole algorithm.
> Which means: total while-loop iterations (total removals) across the entire run `≤ n`, no matter how they clump into individual `r` steps.

Now put it together: the outer for-loop runs exactly `n` times, and the total inner while-loop work across all `n` outer iterations combined is also `≤ n`. What's the total operation count for the whole algorithm, and what complexity does that give you? (This is called **amortized/aggregate analysis** — you bound the _total_ cost over the whole run, not the cost of each individual step.)

And notice something about the argument you just made: did it mention the alphabet size — 26 letters, ASCII, unicode, anything — at all?
- The total operation count for the whole algorithm is `n + n`, time complexity it gives is $O(n)$ amortized, instead of $O(n^2)$. The while loop makes `<= n` operations moving merely forward, so it adds up to outer loop `n` operations making `n + n` which is $O(n)$ in total.
And there is no matter the binding to alphabet size like ASCII, unicode or anything like this.

Exactly right. Total work is `n` (outer loop) + at most `n` (total left-pointer movement across the _whole_ run) = `2n` = $O(n)$ — and that argument never touched alphabet size at all. It holds identically whether the string is 26 lowercase letters or unbounded unicode.

So to close the loop on your earlier idea: the "window can't exceed alphabet size" observation is _true_, but it's not what makes this algorithm O(n) — it's a coincidental extra fact that happens to also be true for small fixed alphabets. The real, general reason is the monotonic two-pointer argument you just derived, which needs nothing about the character set. Good instinct to push on it, but now you've got the actual mechanism, not a special-case one.

> This pattern — bounding total pointer movement across the whole run instead of per-iteration — is called **amortized analysis**, and it's exactly what made the $O(n)$ claim in your sliding window sessions valid, even when individual steps look uneven. File that alongside the "average $O(1)$, worst $O(n)$" caveat you nailed for hash maps earlier — same instinct: don't trust the local/per-step story, check the total.

#### Sliding Window
> Shrink from the left repeatedly until the duplicate is gone, then add the new char.
```py
while string[r] in cs:
    cs.remove(string[l])  # remove from the LEFT, not the duplicate itself
    l += 1
```
Trace on "abba":
```shell
r=0 'a': set={}, add → {a}
r=1 'b': set={a}, add → {a,b}
r=2 'b': 'b' in set! shrink: remove 'a' (l=0→1), still 'b' in set? remove 'b' (l=1→2). Now add 'b' → {b}
r=3 'a': not in set, add → {a,b}
```

In [3]:
def longest_substring(string: str) -> int:
    # setup:
    # keep track of substring len: res = 0
    # store substring chars in window set: char_window = set()
    # set l pointer to 0
    # scan string - for loop full pass

    # 1. add char to char_window
    # 2. update res with: max(res, r - l + 1)
    # 3. return res
    # 4. handle duplicates: while s[r] in char_window: remove s[l] from char_window, move l += 1

    window: set = set()
    res: int = 0
    l: int = 0

    for r in range(len(string)):
        # duplicate handling
        while string[r] in window:
            # shrink from the left repeatedly until the duplicate is gone
            window.remove(string[l])
            l += 1
        # add the new char
        window.add(string[r])
        res = max(res, r - l + 1)

    return res

In [4]:
longest_substring("abcddabcd")

4

> Using **set** data structure (`cs = set()`) for **membership testing** is extremely **efficient**, far more efficient than testing in a list.

### Mental model

When `r` lands on a character already in `window`, the `while` loop removes characters from the left **one at a time** until that specific duplicate is gone.

`window` always holds exactly the characters in the CURRENT WINDOW `[l, r]`. When `r` hits a duplicate, that duplicate's earlier copy is SOMEWHERE in the window. `l` must walk forward past that earlier copy to evict it — removing every character along the way, because they're all still "in front of" `l`.

> **Why this is still $O(n)$ overall:** even though a single `r` step can trigger multiple removals, every character is only ever removed once, ever, across the whole algorithm. Total removals over the entire run `≤ n`. That's the amortized argument.

---

> **`l` only moves forward, and it can never pass `r`.** Across the _entire_ run of the algorithm, `l` moves at most `n` times total — not per iteration, total.

That's the same amortized argument as the min-subarray-sum problem. The `for` loop runs n times. The `while` loop, summed across _all_ iterations of the for loop, also runs at most n times — because `l` has nowhere to go but forward, and it's capped at n.

```shell
Total work = n (r's movements) + n (l's total movements, capped) = O(n)
```

> It does **not** degrade to $O(n²)$ under any input, including all-duplicates. That's the whole point of two-pointer/sliding-window **amortized analysis** — fix this in memory, it'll come up again.

**`longest_substring2` (real sliding window)**:
- Time $O(n)$
- Space $O(min(n, alphabet\space size))$ — set holds at most one window's worth of unique chars

**`longest_substring`  (brute force)**:
- Time $O(n²)$
- Space $O(n)$ — fresh set per outer iteration

```shell
"pwwkew"
 012345
```

```shell
right=0 'p': not in set → add.                     set={p}              left=0   window="p"      len=1
right=1 'w': not in set → add.                     set={p,w}            left=0   window="pw"     len=2
right=2 'w': DUPLICATE → shrink:
              remove string[0]='p' → left=1        set={w}
              'w' STILL in set → shrink again:
              remove string[1]='w' → left=2        set={}
              now add 'w'                          set={w}              left=2   window="w"      len=1
right=3 'k': not in set → add.                     set={w,k}            left=2   window="wk"     len=2
right=4 'e': not in set → add.                     set={w,k,e}          left=2   window="wke"    len=3
right=5 'w': DUPLICATE (the 'w' from right=2 still in set) → shrink:
              remove string[2]='w' → left=3        set={k,e}
              'w' no longer in set → stop
              now add 'w'                          set={k,e,w}          left=3   window="kew"    len=3

result = 3
```

---

### Hash map version

In [ ]:
def length_of_longest_substring(s):
    # hash map
    last_seen = {}    # char → last index where it appeared
    left   = 0
    result = 0

    for right, char in enumerate(s):
        if char in last_seen and last_seen[char] >= left:
            left = last_seen[char] + 1    # jump left past the repeat
        last_seen[char] = right           # remember latest position
        result = max(result, right - left + 1)

    return result

In [ ]:
# length_of_longest_substring("abcabcbb")
# length_of_longest_substring("bbbbb")
length_of_longest_substring("pwwkew")